# 05 · Un harness in miniatura

Un **harness** è tutto ciò che circonda il modello per renderlo un agente affidabile.
Ne costruiamo una versione minima con quattro meccanismi che convivono:
1. **audit** delle chiamate ai tool (middleware);
2. **limite** al numero di tool call (sicurezza deterministica);
3. **approvazione umana** prima di un'azione sensibile;
4. **verifica** del risultato prima di concludere.

## Setup (autonomo)

Ogni notebook è **indipendente**: non importa nulla dal progetto. Qui carichiamo la chiave
API dal file `.env` e creiamo un modello. Esegui le celle in ordine dall'alto verso il basso.

In [ ]:
# Carichiamo le variabili d'ambiente dal file `.env`.
# Lo cerchiamo nella cartella corrente e in quelle superiori, così il notebook
# funziona sia se avviato dalla radice del progetto sia dalla cartella `notebooks`.
import os
from pathlib import Path

from dotenv import load_dotenv


def trova_env() -> Path:
    for cartella in (Path.cwd(), *Path.cwd().resolve().parents):
        if (cartella / ".env").is_file():
            return cartella / ".env"
    raise FileNotFoundError("File .env non trovato: copia .env.example in .env e aggiungi la chiave.")


env_file = trova_env()
load_dotenv(env_file, override=False)          # carica le variabili senza sovrascrivere quelle già presenti
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY mancante nel file .env"
print("Ambiente caricato da:", env_file)

In [ ]:
# `ChatOpenAI` è il wrapper LangChain attorno al modello.
# Lo creiamo una volta e lo riusiamo in tutto il notebook.
from langchain_openai import ChatOpenAI

MODELLO = os.getenv("OPENAI_MODEL", "gpt-5.4-mini")   # modello economico, va bene per imparare
model = ChatOpenAI(
    model=MODELLO,
    use_responses_api=True,   # API "responses" di OpenAI
    store=False,              # non conservare la conversazione sui server OpenAI
)
print("Modello pronto:", MODELLO)

## 1 · Un tool con effetto collaterale

Simuliamo un'azione "sensibile": inviare una email. Non la spediamo davvero, la registriamo.

In [ ]:
from langchain_core.tools import tool

INVIATE: list[dict] = []


@tool
def invia_email(destinatario: str, testo: str) -> str:
    """Invia una email (qui simulata) a un destinatario."""
    INVIATE.append({"a": destinatario, "testo": testo})
    return f"Email inviata a {destinatario}."

## 2 · Audit: un middleware attorno ai tool

Un middleware `AgentMiddleware` può avvolgere ogni chiamata a un tool. Lo usiamo per
**tracciare** cosa viene eseguito — senza registrare argomenti sensibili.

In [ ]:
from langchain.agents.middleware import AgentMiddleware

TRACCIA: list[str] = []


class AuditMiddleware(AgentMiddleware):
    # `wrap_tool_call` viene chiamato per ogni tool: prima, dopo (o su errore).
    def wrap_tool_call(self, request, handler):
        nome = request.tool_call["name"]
        TRACCIA.append(f"start:{nome}")
        risultato = handler(request)      # esegue davvero il tool
        TRACCIA.append(f"done:{nome}")
        return risultato

## 3 · Limite di tool call

Un tetto deterministico evita loop infiniti o costi fuori controllo. LangChain offre un
middleware pronto: si ferma quando l'agente supera il numero di chiamate consentite.

In [ ]:
from langchain.agents.middleware import ToolCallLimitMiddleware

limite = ToolCallLimitMiddleware(run_limit=5, exit_behavior="end")   # max 5 tool call per run

## 4 · Approvazione umana (human-in-the-loop)

Per le azioni sensibili vogliamo che un umano approvi. Lo mostriamo in modo esplicito:
eseguiamo l'agente in *modalità simulazione* e vediamo cosa AVREBBE fatto, decidendo noi.
(LangChain supporta anche interruzioni automatiche con `interrupt_on`.)

In [ ]:
from langchain.agents import create_agent

agente = create_agent(
    model=model,
    tools=[invia_email],
    middleware=[AuditMiddleware(), limite],   # audit + limite attivi
    system_prompt="Quando l'utente chiede di scrivere a qualcuno, usa invia_email.",
)

In [ ]:
esito = agente.invoke({"messages": [{
    "role": "user",
    "content": "Scrivi a mario@example.com un breve promemoria per la riunione di domani.",
}]})
print(esito["messages"][-1].text)

## 5 · Verifica del risultato

Prima di fidarci, controlliamo con del **codice deterministico** che l'effetto ci sia stato.
La verifica non la fa il modello: la fa il nostro programma.

In [ ]:
# Traccia dei tool e prova concreta dell'effetto collaterale.
print("Traccia audit:", TRACCIA)
print("Email registrate:", INVIATE)
assert len(INVIATE) == 1, "Attesa esattamente una email"
print("Verifica superata: l'azione è avvenuta una sola volta.")

## Prova tu

- Abbassa `run_limit` a 1 e chiedi due email: vedrai il limite fermare l'agente.
- Nell'audit, aggiungi la durata di ogni tool (con `time.monotonic()`).

**Idea chiave**: l'harness è un insieme di *guardrail* attorno al modello — osservabilità,
limiti, approvazione e verifica — che trasformano un modello in un agente su cui fare affidamento.